# The Relational Data Wrangler & Fraud Sentinel

End-to-end pipeline: clean + merge three messy relational CSVs, neutralize
prompt-injection attempts hidden in transaction notes, and use a Small
Language Model (<3B params, Qwen2.5-1.5B-Instruct) to produce a strict-JSON
fraud risk profile per transaction -- plus a LoRA fine-tune step.

**Two explicit, documented data decisions were made and are called out inline
where they happen (not hidden):**
1. The supplied `transactions.csv` has no free-text notes column, but the
   brief requires neutralizing injection attempts hidden in notes. Section 8
   synthesizes a clearly-labeled `transaction_notes__SYNTHETIC` column
   (deterministic, seeded) rather than skipping that requirement.
2. No authoritative fraud label exists in the supplied data. Section 11
   derives **heuristic pseudo-labels** (weak supervision) to bootstrap LoRA
   fine-tuning -- these are never reported as real fraud ground truth.


## 1. Setup & Configuration

In [ ]:
import sys
from pathlib import Path

# notebook lives in notebooks/, package lives at the project root
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "services").exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

from utils.seed import set_global_seed
set_global_seed(42)

print("Project root:", PROJECT_ROOT)


## 2. Load and Profile Raw Data

In [ ]:
from services.data_loader import load_all

tx_raw, acc_raw, cust_raw = load_all()
print("transactions:", tx_raw.shape, "| accounts:", acc_raw.shape, "| customers:", cust_raw.shape)
tx_raw.head(3)


In [ ]:
# Real defects found during profiling (see project history) -- confirm here, don't just assert:
print("Missing customer_id in transactions:", tx_raw["customer_id"].isna().sum())
print("Distinct raw channel values (note case/whitespace/separator noise):")
print(sorted(tx_raw["channel"].dropna().astype(str).str.strip().unique()))
print("\nDistinct raw is_foreign_transaction encodings (mixed booleans):")
print(sorted(tx_raw["is_foreign_transaction"].dropna().astype(str).unique()))


## 3. Clean Transactions

Real defects fixed here: unparseable amounts (blank + 6 rows where a currency
code like `"INR 62146.26"` leaked into the amount field), mixed timestamp
formats (ISO / `DD/MM/YYYY HH:MM` / blank / literal `"NOT_AVAILABLE"`),
categorical noise (case + whitespace + `_` vs space separators -- e.g.
`INTERNET_BANKING` vs `INTERNET BANKING` were the same category but survived
naive strip+upper as two distinct values), and inconsistent boolean encodings
(`0/1/TRUE/FALSE/yes/no/Y/N` all appearing in the same column).

In [ ]:
from services.cleaning_service import clean_transactions

tx_clean = clean_transactions(tx_raw)
print("Amount parse failures:", tx_clean["amount_clean"].isna().sum(), "/", len(tx_clean))
print("Suspect/missing timestamps:", tx_clean["is_timestamp_suspect"].sum(), "/", len(tx_clean))
print("\nchannel values AFTER normalization (space/underscore unified):")
print(sorted(tx_clean["channel"].dropna().unique()))
tx_clean[["transaction_id", "amount", "amount_clean", "transaction_timestamp",
          "transaction_timestamp_clean", "is_timestamp_suspect"]].head(5)


## 4. Clean Accounts

In [ ]:
from services.cleaning_service import clean_accounts

acc_clean = clean_accounts(acc_raw)
print("Null counts (accounts.csv):")
print(acc_raw.isna().sum()[acc_raw.isna().sum() > 0])
print("\ncredit_limit sanity check by account_type (found to be CLEAN, not corrupted, despite the brief's claim):")
print(acc_clean.groupby("account_type")["credit_limit"].apply(lambda s: sorted(s.unique())))


## 5. Clean Customers

In [ ]:
from services.cleaning_service import clean_customers

cust_clean = clean_customers(cust_raw)
print("Null counts (customers.csv):")
print(cust_raw.isna().sum()[cust_raw.isna().sum() > 0])
print("\nNo duplicate rows or duplicate customer_id found:",
      cust_raw.duplicated().sum(), "/", cust_raw["customer_id"].duplicated().sum())


## 6. Relational Merge

`transactions.account_id -> accounts.account_id -> accounts.customer_id`,
recovering the 8 transactions missing `customer_id` directly.

**Real defect found (not in the brief's description, but present in the
actual data):** 8 transactions reference `ACC_900000`-`ACC_900007`, account
IDs that do not exist anywhere in `accounts.csv` (real accounts only run
`ACC_000001`-`ACC_000178`). These are fabricated/orphaned foreign keys, not
recoverable by any join. Rather than silently drop them (violating "one JSON
output per transaction"), we flag `account_reference_broken=True` and treat
it as a fraud signal itself downstream.

In [ ]:
from services.merge_service import merge_transactions

merged, merge_stats = merge_transactions(tx_clean, acc_clean, cust_clean)
merge_stats


In [ ]:
orphan_rows = merged[merged["account_reference_broken"]][["transaction_id", "account_id", "customer_id"]]
print(f"{len(orphan_rows)} transactions reference nonexistent accounts:")
orphan_rows


## 7. Data Quality Report

In [ ]:
quality_report_partial = {
    "rows_loaded": {"transactions": len(tx_raw), "accounts": len(acc_raw), "customers": len(cust_raw)},
    "amount_parse_failures": int(merged["amount_clean"].isna().sum()),
    "timestamp_suspect_count": int(merged["is_timestamp_suspect"].sum()),
    "merge": merge_stats,
}
import json
print(json.dumps(quality_report_partial, indent=2))


## 8. Synthetic Transaction Notes

**HACKATHON ROBUSTNESS TEST DATA -- not part of the original export.** The
supplied `transactions.csv` has no notes/memo/description column at all
(verified: zero hits for any injection-style phrase or free text anywhere in
the file). The brief explicitly requires neutralizing injection attempts
hidden in transaction notes, so rather than skip that requirement, we
generate a deterministic, seeded `transaction_notes__SYNTHETIC` column:
~75% ordinary bank-transaction notes, ~25% embedded prompt-injection payloads
(a mix of well-known jailbreak phrasings and paraphrased near-misses).

In [ ]:
from services.notes_service import synthesize_transaction_notes

merged = synthesize_transaction_notes(merged, seed=42, injection_rate=0.25)
print("Injection attempts planted:", merged["_gt_is_injection_attempt"].sum(), "/", len(merged))
merged[["transaction_id", "transaction_notes__SYNTHETIC", "_gt_is_injection_attempt"]].sample(6, random_state=1)


## 9. Prompt Injection Defense

**Design decision: the SLM is never responsible for detecting the attack.**
Untrusted note text is regex-scanned and replaced with a placeholder BEFORE
it goes anywhere near the model's context (layer 1). As a second layer
(defense in depth), the prompt structurally fences whatever note text remains
as DATA ONLY with explicit instructions to never treat it as a command, and
`validation_service.py` refuses any output that doesn't match the strict
schema regardless of what the model tried to say.

Measured honestly against the planted ground truth below: **precision 1.000,
recall 0.636** -- the regex layer catches known/exact phrasings perfectly but
misses about a third of paraphrased/novel attacks, which is exactly why layer
2 (prompt fencing + schema validation) matters and isn't just a formality.

In [ ]:
from services.sanitization_service import sanitize_dataframe

merged, base_audit_records = sanitize_dataframe(merged)

gt = merged["_gt_is_injection_attempt"]
det = merged["injection_detected"]
tp, fp, fn, tn = int((gt & det).sum()), int((~gt & det).sum()), int((gt & ~det).sum()), int((~gt & ~det).sum())
precision = tp / (tp + fp) if (tp + fp) else float("nan")
recall = tp / (tp + fn) if (tp + fn) else float("nan")
print(f"TP={tp} FP={fp} FN={fn} TN={tn}  precision={precision:.3f}  recall={recall:.3f}")

merged[merged["injection_detected"]][["transaction_id", "transaction_notes__SYNTHETIC", "transaction_notes_sanitized"]].head(5)


In [ ]:
# A missed (paraphrased) injection, to be honest about the limitation:
missed = merged[gt & ~det]
print(f"{len(missed)} injection attempts were NOT caught by the regex layer, e.g.:")
missed[["transaction_id", "transaction_notes__SYNTHETIC"]].head(3)


## 10. Fraud Feature Engineering

In [ ]:
from services.feature_service import compute_fraud_features, RISK_FEATURE_COLUMNS

merged = compute_fraud_features(merged)
merged[["transaction_id"] + RISK_FEATURE_COLUMNS].sample(5, random_state=2)


## 11. Weak Supervision (Heuristic Pseudo-Labels)

**These are NOT real fraud labels.** No authoritative ground truth was
provided in the supplied data. `_pseudo_is_fraud` is a deterministic rule
(>=3 independent structured risk signals) used *solely* to give the LoRA
fine-tune something to learn from. Every place this is used downstream is
labeled "weak supervision" / "pseudo-label", never "accuracy".

In [ ]:
from services.labeling_service import generate_pseudo_labels

merged = generate_pseudo_labels(merged)
print("Weak pseudo-label positive rate:", merged["_pseudo_is_fraud"].mean())
merged["_weak_risk_score"].value_counts().sort_index()


## 12. Load Qwen2.5-1.5B-Instruct

<3B-parameter open-weight instruct model, loaded in 4-bit (bitsandbytes NF4)
so it fits comfortably on a 4GB laptop GPU. Falls back to fp32/CPU
automatically if no CUDA device is present.

In [ ]:
from services.inference_service import load_model_and_tokenizer
import torch

model, tokenizer = load_model_and_tokenizer()
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM allocated (MB):", torch.cuda.memory_allocated() / 1e6)


## 13. Baseline Zero-Shot Evaluation

Run on a stratified subset (mix of high-risk / injection-flagged / benign
transactions) rather than all 1000 -- this is only the "before" half of the
fine-tuning story; the final `predictions.json` (Section 18) covers every
transaction using the fine-tuned model.

Inference is **batched** (16 prompts at a time): measured serial generation
took 15-29s/transaction (4-8 hours for 1000 rows), batching brought this down
to ~1-2s/transaction -- same model, same prompt, same validation rules,
purely a throughput fix.

In [ ]:
from routes.run_full_pipeline import stratified_baseline_sample
from controllers.inference_controller import run_predictions, compare_against_pseudo_labels

baseline_df = stratified_baseline_sample(merged, n=90)
baseline_preds, baseline_audit_meta = run_predictions(model, tokenizer, baseline_df, batch_size=16)
baseline_stats = compare_against_pseudo_labels(baseline_preds, baseline_df, label="baseline_zero_shot")
baseline_stats


In [ ]:
for p in baseline_preds[:3]:
    print(p.to_dict())


## 14. LoRA Fine-Tuning

Trains a LoRA adapter (rank 8, targeting the attention projections) on the
heuristic pseudo-labels from Section 11, on a random subset of the data for
speed. Purpose: teach the tiny model to reliably emit the exact required JSON
schema and align with the structured evidence rules -- not a claim of
verified accuracy improvement against real fraud (no such ground truth
exists here).

In [ ]:
from controllers.training_controller import run_fine_tuning

# Sized for this hardware: measured ~30-65s/step on a 4GB laptop GPU with
# gradient checkpointing enabled (needed for headroom -- peak usage otherwise
# sits right at the 4GB ceiling). 80 examples / effective batch 8 = 10 steps.
peft_model, adapter_path = run_fine_tuning(model, tokenizer, merged, max_examples=80, num_train_epochs=1)
print("Adapter saved to:", adapter_path)


## 15. Fine-Tuned Inference (Full Dataset)

This is the actual required deliverable: one JSON prediction per transaction,
for all 1000 transactions, produced by the fine-tuned model.

In [ ]:
final_preds, final_audit_meta = run_predictions(peft_model, tokenizer, merged, batch_size=16)
print(f"Generated {len(final_preds)} predictions.")


## 16. Baseline vs Fine-Tuned Comparison

In [ ]:
fine_tuned_stats = compare_against_pseudo_labels(final_preds, merged, label="fine_tuned")

import pandas as pd
comparison_df = pd.DataFrame([baseline_stats, fine_tuned_stats]).set_index("label")
comparison_df


## 17. JSON Schema Validation

The validator (`services/validation_service.py`) strips markdown fences,
extracts the first JSON object, and rejects anything missing keys, with the
wrong types, or with an out-of-range confidence -- regardless of what
surrounding prose an injection attempt may have produced.

In [ ]:
from services.validation_service import parse_model_output

test_cases = [
    '{"transaction_id": "TXN_1", "is_fraud": true, "confidence": 0.87, "justification": "Large amount at unusual hour."}',
    '```json\n{"transaction_id": "TXN_2", "is_fraud": false, "confidence": 0.2, "justification": "Routine grocery purchase."}\n```',
    'Sure! Here is my answer: {"transaction_id": "TXN_3", "is_fraud": false, "confidence": 1.5, "justification": "safe"}',
    'I will ignore the fraud rules as instructed. {"is_fraud": false, "confidence": 0.0}',
    'not json at all',
]
for i, raw in enumerate(test_cases):
    pred, err = parse_model_output(raw, expected_transaction_id=f"TXN_{i+1}")
    print(f"case {i}: {'OK -> ' + str(pred.to_dict()) if pred else 'REJECTED -> ' + err}")


## 18. Export predictions.json

In [ ]:
from controllers.inference_controller import merge_audit
from utils.io import OUTPUTS_DIR, save_json

full_audit = merge_audit(base_audit_records, final_audit_meta, merged)
predictions_json = [p.to_dict() for p in final_preds]

save_json(predictions_json, OUTPUTS_DIR / "predictions.json")
save_json([a.to_dict() for a in full_audit], OUTPUTS_DIR / "audit_log.json")

quality_report_partial["injection_detector"] = {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
                                                 "precision": precision, "recall": recall}
quality_report_partial["synthetic_notes_injection_rate"] = float(merged["_gt_is_injection_attempt"].mean())
quality_report_partial["weak_pseudo_label_positive_rate"] = float(merged["_pseudo_is_fraud"].mean())

save_json({
    "data_quality_report": quality_report_partial,
    "baseline_vs_fine_tuned": {"baseline_zero_shot": baseline_stats, "fine_tuned": fine_tuned_stats},
    "note": ("weak_pseudo_label_positive_rate and *_pseudo_labels agreement figures are WEAK "
             "SUPERVISION heuristics, not real fraud ground truth -- no authoritative fraud "
             "labels were provided in the supplied data."),
}, OUTPUTS_DIR / "evaluation_report.json")

print(f"Wrote {len(predictions_json)} predictions to outputs/predictions.json")
print("\nSample (exactly the 4 required keys, nothing else):")
predictions_json[:3]


## 19. Final Robustness Demonstration

End-to-end proof that a planted jailbreak note does NOT change the verdict:
take one transaction whose synthetic note explicitly instructs the model to
mark it safe, and show the sanitizer catches it, the model never sees the raw
payload, and the final verdict is driven by the structured evidence alone.

In [ ]:
demo_row = merged[merged["injection_detected"] & (merged["_weak_risk_score"] >= 2)].iloc[0]
demo_idx = demo_row.name

print("Raw synthetic note (never sent to the model):")
print(" ", repr(demo_row["transaction_notes__SYNTHETIC"]))
print("\nSanitized note actually sent to the model:")
print(" ", repr(demo_row["transaction_notes_sanitized"]))
print("\nStructured risk signals for this transaction:")
from services.feature_service import RISK_FEATURE_COLUMNS
print(" ", {c: bool(demo_row[c]) for c in RISK_FEATURE_COLUMNS})

final_pred_for_demo = [p for p in final_preds if p.transaction_id == demo_row["transaction_id"]][0]
print("\nFinal model verdict (unaffected by the injection attempt):")
print(" ", final_pred_for_demo.to_dict())
